# ShopDesk, Module 2 Section 3 Lab 2: Read+Write Fallback and Incremental Exploration

A beginner-friendly notebook on two habits that keep code work safe and cheap: falling back to
**Read then Write** when a targeted **Edit** cannot be made unique, and exploring a codebase
**incrementally** (Grep to Read to trace) instead of loading all of it. Pure-Python analogues make
both concrete offline; a live **Claude Agent SDK** run uses the real tools. Runs **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

Sometimes a change touches text that appears several times, so a single-anchor Edit refuses. And
sometimes you just need to understand how ShopDesk's refund path works before touching it. The
answers are a **Read plus Write** fallback for the edit, and an **incremental** exploration that
reads only the slices it needs, keeping the context window small.

The question this lab answers: **what do you do when Edit cannot be unique, and how do you explore a
codebase without drowning in it?**

## Objectives

- Fall back to **Read then modify then Write** when an Edit anchor is ambiguous.
- Explore **incrementally**: Grep to find, Read the slice, trace the call to the next piece.
- Keep context small by reading only what you need, not the whole codebase.

## What you'll observe

- An ambiguous Edit fails, and the Read+Write fallback applies the change anyway.
- An incremental trace follows `process_refund` to `is_refundable` to `_lookup`, reading only those
  slices.
- The incremental reads total a small fraction of the whole repo's bytes.

## How to run

Run top to bottom. The repo creation, the fallback, and the exploration cells run anywhere. The live
cell calls Claude with the real built-in tools, so paste a real key into **Setup 2/3** and re-run
from the top; otherwise it skips. **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. The live cell uses the **Agent SDK** to drive Claude
Code's built-in tools; the offline cells use only Python. The Agent SDK also needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the Agent SDK =====
%pip install -q claude-agent-sdk anthropic python-dotenv

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live
cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sample repo
import re                                       # our offline Grep uses regex
import sys                                       # detect Windows (it needs a special event loop)
from pathlib import Path                          # walk the repo
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** writes the same small **ShopDesk sample repo** as the previous lab, so this one
stands alone. The refund path (`process_refund` calls `is_refundable` calls `_lookup`) is what we will
trace and edit.

In [ ]:
# ===== SETUP 3/3 - create the sample repo =====
import textwrap                                    # keeps the embedded file bodies readable
REPO = os.path.join(os.getcwd(), "shopdesk_repo2") # a fresh sample codebase root

FILES = {
    "shopdesk/__init__.py": "",
    "shopdesk/orders.py": textwrap.dedent("""\
        ORDERS = {
            "A1": {"status": 2, "delivered_days_ago": 5},
            "A2": {"status": 3, "delivered_days_ago": 60},
        }
        STATUS_NAMES = {1: "processing", 2: "shipped", 3: "delivered"}

        def get_order_status(order_id):
            order = _lookup(order_id)
            if order is None:
                return None
            return STATUS_NAMES[order["status"]]

        def _lookup(order_id):
            return ORDERS.get(order_id)
        """),
    "shopdesk/refunds.py": textwrap.dedent("""\
        from shopdesk.orders import _lookup

        REFUND_WINDOW_DAYS = 30

        def is_refundable(order_id):
            order = _lookup(order_id)
            if order is None:
                return None
            return order["delivered_days_ago"] <= REFUND_WINDOW_DAYS

        def process_refund(order_id):
            eligible = is_refundable(order_id)
            if eligible is None:
                return None
            return "refunded " + order_id if eligible else "refused: outside window"
        """),
    "shopdesk/shipping.py": 'def get_tracking(order_id):\n    return "https://track.example/" + order_id\n',
    "tests/test_orders.py": 'from shopdesk.orders import get_order_status\ndef test_status():\n    assert get_order_status("A1") == "shipped"\n',
    "tests/test_refunds.py": 'from shopdesk.refunds import process_refund\ndef test_refund():\n    assert process_refund("A1").startswith("refunded")\n',
    "config/settings.toml": '[shopdesk]\nrefund_window_days = 30\n',
    "pyproject.toml": '[tool.pytest.ini_options]\ntestpaths = ["tests"]\n',
    "README.md": "# ShopDesk sample repo\n",
}
for rel, content in FILES.items():                 # write every file
    path = os.path.join(REPO, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("created repo at", REPO, "with", len(FILES), "files")

**This cell:** the offline **Grep**, **Read**, **Edit**, and **Write** analogues we will use.
`edit()` enforces the uniqueness rule (so it can fail), and `write_file()` overwrites a whole file,
which is what the fallback relies on.

In [ ]:
# ===== the offline tool analogues =====
def grep(pattern, file_glob="**/*.py"):            # content search -> (path, line_no, text)
    rx = re.compile(pattern); hits = []
    for p in Path(REPO).glob(file_glob):
        if p.is_file():
            for i, line in enumerate(p.read_text().splitlines(), 1):
                if rx.search(line): hits.append((str(p.relative_to(REPO)), i, line.strip()))
    return hits

def read_file(rel, start=1, end=None):             # read a slice, return the text (no line numbers)
    lines = open(os.path.join(REPO, rel)).read().splitlines()
    end = end or len(lines)
    return "\n".join(lines[start - 1:end])

def edit(rel, old, new):                           # targeted edit; fails unless the anchor is unique
    text = open(os.path.join(REPO, rel)).read()
    n = text.count(old)
    if n != 1: return f"FAILED: anchor appears {n} times in {rel} (need exactly 1)"
    open(os.path.join(REPO, rel), "w").write(text.replace(old, new)); return f"OK: edited {rel}"

def write_file(rel, content):                      # overwrite a whole file (the fallback's tool)
    open(os.path.join(REPO, rel), "w").write(content); return f"OK: wrote {rel}"
print("tools ready: grep, read_file, edit, write_file")

---

### 🎯 Lab objective - fall back gracefully, explore incrementally

**What you build:** a Read+Write fallback for an ambiguous edit, and an incremental trace of the
refund path with a context-cost comparison.

**Why it helps you build real solutions:** you will hit anchors that cannot be made unique, and real
repos are far too big to load whole. These two habits keep edits safe and context lean.

**How you'll see it:** the fallback applies a change Edit refused, and the incremental trace reads a
fraction of the repo.

**This cell:** the setup for the fallback. We try a targeted **Edit** on `return None`, which
appears twice in `refunds.py`, so Edit refuses. When you cannot make the anchor unique (and do not
want a per-occurrence edit), this is the moment to fall back.

In [ ]:
# ===== Edit refuses an ambiguous anchor =====
print("'return None' count:", open(os.path.join(REPO, "shopdesk/refunds.py")).read().count("return None"))
print(edit("shopdesk/refunds.py", "return None", 'return "unknown order"'))   # -> FAILED (2 matches)

**This cell:** the **Read then modify then Write** fallback. We Read the whole file, transform it
in memory (here, replace every `return None` at once), then Write it back. This handles the change Edit
could not, and it is the right tool for multi-occurrence or structural edits.

In [ ]:
# ===== fallback: Read -> modify in memory -> Write =====
text = read_file("shopdesk/refunds.py")            # READ the whole file
text = text.replace("return None", 'return "unknown order"')   # MODIFY (all occurrences) in memory
print(write_file("shopdesk/refunds.py", text + "\n"))          # WRITE it back
print("'return None' count now:", open(os.path.join(REPO, "shopdesk/refunds.py")).read().count("return None"))

**This cell:** an **incremental exploration** of the refund path. We Grep for the entry point,
Read just that function, notice it calls `is_refundable`, Read that, notice it calls `_lookup`, and
Grep+Read that, tracing the logic one slice at a time. We never open the tests, config, or README.

In [ ]:
# ===== trace process_refund -> is_refundable -> _lookup, one slice at a time =====
read_bytes = 0                                     # count only what we actually read

def first_hit(pattern):                             # helper: the file+line of the first grep match
    hits = grep(pattern); return hits[0] if hits else None

f, line, _ = first_hit(r"def process_refund")       # STEP 1: find the entry point
slice1 = read_file(f, line, line + 3); read_bytes += len(slice1)   # STEP 2: read just that function
print("process_refund in", f, "\n", slice1, "\n")

f2, line2, _ = first_hit(r"def is_refundable")      # STEP 3: it calls is_refundable -> find it
slice2 = read_file(f2, line2, line2 + 4); read_bytes += len(slice2)   # read that slice
print("is_refundable in", f2, "\n", slice2, "\n")

f3, line3, _ = first_hit(r"def _lookup")            # STEP 4: it calls _lookup -> find it
slice3 = read_file(f3, line3, line3 + 1); read_bytes += len(slice3)   # read that slice
print("_lookup in", f3, "\n", slice3)

**This cell:** the **context-cost** comparison. We add up the bytes we read incrementally and
compare them to the bytes of the whole repo. Reading only the slices you need is what keeps the context
window lean on a real codebase.

In [ ]:
# ===== incremental reads vs loading everything =====
full_bytes = sum(len(p.read_text()) for p in Path(REPO).rglob("*") if p.is_file())   # the whole repo
print("incremental bytes read:", read_bytes)
print("whole repo bytes:      ", full_bytes)
print(f"read {read_bytes / full_bytes:.0%} of the repo to trace the refund path")

**This cell:** the live run with the **real built-in tools**, allowing only `Grep`, `Read`, and
`Write` (no `Edit`). Ask the agent to trace and adjust the refund path; it will Grep and Read
incrementally, and use Write for the change since Edit is not available.

In [ ]:
# ===== live: Grep + Read to explore, Write to change =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock

EXPLORE_OPTS = ClaudeAgentOptions(                  # point the tools at our sample repo
    model=MODEL, cwd=REPO,
    allowed_tools=["Grep", "Read", "Write"])        # note: no Edit here, so Write is the fallback

async def ask(prompt):                              # stream the run and show each tool call
    async for m in query(prompt=prompt, options=EXPLORE_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name, {k: b.input[k] for k in list(b.input)[:2]})
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:140])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: ask("Trace how process_refund decides eligibility, then make both 'return None' cases return 'unknown order'."))
else:
    print("[skipped] expected: Grep + Read trace the refund path, then Write updates refunds.py")
    print("          (Edit is not allowed, so the agent uses the Read+Write fallback).")

| anti-pattern | what to do instead |
|---|---|
| force a per-occurrence Edit on a repeated anchor | Read the file, transform in memory, Write it back |
| load the whole repo before starting | Grep to find, Read only the slice you need |
| re-Read a file you already have | keep the slice; only Read again if it changed |
| use Write for a one-line change on a unique anchor | prefer Edit; reserve Write for the fallback |

**Lesson:** when an anchor cannot be unique, fall back to **Read plus Write**: load the file,
change it in memory, write it back. And explore **incrementally**, Grep to Read to trace, so you pull
in only the context the task needs. Both habits keep changes safe and the context window small on a
real codebase.

---

## Recap - fallback and incremental exploration

| Situation | Move | Why |
|---|---|---|
| Edit anchor ambiguous | Read then Write | change what Edit cannot do safely |
| Need to understand code | Grep then Read then trace | follow the logic one slice at a time |
| Large codebase | read only needed slices | keep the context window lean |

One principle to carry forward: **prefer the surgical Edit, fall back to Read plus Write when you must,
and never load more of the codebase than the task needs.** To run live, paste a real key into
**Setup 2/3** and re-run from the top. Then try it: add a fourth module to the refund path and watch the
incremental trace extend by exactly one slice.